
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



# LAB: Batch Inference Using SLM

In this lab, you will learn how to implement a batch inference pipeline using a Small Language Model (SLM) in a production environment. The objective is to follow a structured approach to develop, test, and deploy a language model-based pipeline using tools such as MLflow, and Unity Catalog. This process focuses on effective model management and operational strategies, facilitating batch inference using Spark DataFrames, and managing model life cycles via model registration and querying.


**Lab Outline:**

*In this lab, you will need to complete the following tasks:*

1. **Task 1:** Create a Hugging Face question-answering pipeline and test it.
2. **Task 2:** Track and register the model using MLflow and Unity Catalog.
3. **Task 3:** Manage the registered model's state.
4. **Task 4:** Perform single-node and multi-node batch inference.
5. **Task 5:** Perform batch inference using SQL `ai_query`.

## REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:
1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

   - Click **More** in the drop-down.
   
   - In the **Attach to an existing compute resource** window, use the first drop-down to select your unique cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

2. Find the triangle icon to the right of your compute cluster name and click it.

3. Wait a few minutes for the cluster to start.

4. Once the cluster is running, complete the steps above to select your cluster.

## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need to use one of the following Databricks runtime(s): **15.4.x-cpu-ml-scala2.12**


## Classroom Setup

Install required libraries.

In [0]:
%pip install -qq -U huggingface-hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tokenizers 0.19.0 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.1.4 which is incompatible.
transformers 4.41.2 requires huggingface-hub<1.0,>=0.23.0, but you have huggingface-hub 1.1.4 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install mlflow>=3.0 databricks-feature-engineering --upgrade
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.11.0 which is incompatible.
langchain 0.1.20 requires langchain-core<0.2.0,>=0.1.52, but you have langchain-core 1.0.5 which is incompatible.
langchain 0.1.20 requires langsmith<0.2.0,>=0.1.17, but you have langsmith 0.4.43 which is incompatible.
langchain 0.1.20 requires tenacity<9.0.0,>=8.1.0, but you have tenacity 9.1.2 which is incompatible.
langchain-community 0.0.38 requires langchain-core<0.2.0,>=0.1.52, but you have langchain-core 1.0.5 which is incompatible.
langchain-community 0.0.38 requires langsmith<0.2.0,>=0.1.0, but you have langsmith 0.4.43 which is incompatible.
langchain-community 0.0.38 requires tenacity<9.0.0,>=8.1.0, but you have tenacity 9.1.2 which is incompatible.
langchain-text-splitters 0.0.2 requires langchain-core<0.3,

Before starting the Lab, run the provided classroom setup script. This script will define configuration variables necessary for the lab. Execute the following cell:

In [0]:
%run ../Includes/Classroom-Setup-01

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
databricks-agents 1.8.2 requires databricks-sdk[openai]>=0.58.0, but you have databricks-sdk 0.36.0 which is incompatible.
databricks-feature-engineering 0.13.0 requires databricks-sdk>=0.62.0, but you have databricks-sdk 0.36.0 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.



The examples and models presented in this course are intended solely for demonstration and educational purposes.
 Please note that the models and prompt examples may sometimes contain offensive, inaccurate, biased, or harmful content.


**Other Conventions:**

Throughout this lab, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")

Username:          labuser12678309_1763243239@vocareum.com
Catalog Name:      dbacademy
Schema Name:       labuser12678309_1763243239
Working Directory: /Volumes/dbacademy/ops/labuser12678309_1763243239@vocareum_com
Dataset Location:  NestedNamespace (arxiv='/Volumes/dbacademy_arxiv/v01')


## Dataset Overview

In this Lab, you will be using the SQuAD dataset hosted on HuggingFace. This is a reading comprehension dataset which consists of questions and answers based on the provided context. Let's load and inspect the structure of the SQuAD dataset.

In [0]:
from datasets import load_dataset
from delta.tables import DeltaTable

prod_data_table_name = f"{DA.catalog_name}.{DA.schema_name}.m4_1_lab_prod_data"
squad_dataset = load_dataset("squad")
test_spark_df = spark.createDataFrame(squad_dataset["validation"].to_pandas())
test_spark_df.write.mode("overwrite").saveAsTable(prod_data_table_name)

/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:45: UserWarning: The cache_dir for this dataset is /root/.cache, which is not a persistent path.Therefore, if/when the cluster restarts, the downloaded dataset will be lost.The persistent storage options for this workspace/cluster config are: [DBFS, UC Volumes].Please update either `cache_dir` or the environment variable `HF_DATASETS_CACHE`to be under one of the following root directories: ['/dbfs/', '/Volumes/']
  warnings.warn(warning_message)


/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:14: UserWarning: During large dataset downloads, there could be multiple progress bar widgets that can cause performance issues for your notebook or browser. To avoid these issues, use `datasets.utils.logging.disable_progress_bar()` to turn off the progress bars.
  warnings.warn(


Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [0]:
spark.table(prod_data_table_name).limit(20).display()

id,title,context,question,answers
56be4db0acb8001400a502ec,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the ""golden anniversary"" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as ""Super Bowl L""), so that the logo could prominently feature the Arabic numerals 50.",Which NFL team represented the AFC at Super Bowl 50?,"List(List(177, 177, 177), List(Denver Broncos, Denver Broncos, Denver Broncos))"
56be4db0acb8001400a502ed,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the ""golden anniversary"" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as ""Super Bowl L""), so that the logo could prominently feature the Arabic numerals 50.",Which NFL team represented the NFC at Super Bowl 50?,"List(List(249, 249, 249), List(Carolina Panthers, Carolina Panthers, Carolina Panthers))"
56be4db0acb8001400a502ee,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the ""golden anniversary"" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as ""Super Bowl L""), so that the logo could prominently feature the Arabic numerals 50.",Where did Super Bowl 50 take place?,"List(List(403, 355, 355), List(Santa Clara, California, Levi's Stadium, Levi's Stadium in the San Francisco Bay Area at Santa Clara, California.))"
56be4db0acb8001400a502ef,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the ""golden anniversary"" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as ""Super Bowl L""), so that the logo could prominently feature the Arabic numerals 50.",Which NFL team won Super Bowl 50?,"List(List(177, 177, 177), List(Denver Broncos, Denver Broncos, Denver Broncos))"
56be4db0acb8001400a502f0,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the c

In [0]:
spark.table(prod_data_table_name).select("title").distinct().display()

title
Harvard_University
Civil_disobedience
Construction
Pharmacy
Victoria_and_Albert_Museum
Immune_system
American_Broadcasting_Company
Genghis_Khan
Private_school
Economic_inequality


In [0]:
spark.table(prod_data_table_name).where(F.col("title")=="Imperialism").distinct().display()

id,title,context,question,answers
57308f6b8ab72b1400f9c581,Imperialism,"Orientalism, as theorized by Edward Said, refers to how the West developed an imaginative geography of the East. This imaginative geography relies on an essentializing discourse that represents neither the diversity nor the social reality of the East. Rather, by essentializing the East, this discourse uses the idea of place-based identities to create difference and distance between ""we"" the West and ""them"" the East, or ""here"" in the West and ""there"" in the East. This difference was particularly apparent in textual and visual works of early European studies of the Orient that positioned the East as irrational and backward in opposition to the rational and progressive West. Defining the East as a negative vision of itself, as its inferior, not only increased the West’s sense of self, but also was a way of ordering the East and making it known to the West so that it could be dominated and controlled. The discourse of Orientalism therefore served as an ideological justification of early Western imperialism, as it formed a body of knowledge and ideas that rationalized social, cultural, political, and economic control of other territories.",Early Western texts referencing the East describe the people as being what?,"List(List(605, 404, 602, 605, 605), List(irrational and backward, them, as irrational and backward, irrational and backward, irrational and backward))"
5730b6592461fd1900a9cfcf,Imperialism,"A resurgence came in the late 19th century, with the Scramble for Africa and major additions in Asia and the Middle East. The British spirit of imperialism was expressed by Joseph Chamberlain and Lord Rosebury, and implemented in Africa by Cecil Rhodes. The pseudo-sciences of Social Darwinism and theories of race formed an ideological underpinning during this time. Other influential spokesmen included Lord Cromer, Lord Curzon, General Kitchner, Lord Milner, and the writer Rudyard Kipling. The British Empire was the largest Empire that the world has ever seen both in terms of landmass and population. Its power, both military and economic, remained unmatched.","By the late 19th century, which country had the largest empire ever to exist in the world?","List(List(494, 498, 122, 498, 498), List(The British Empire, British Empire, The British, British, British))"
5730909d8ab72b1400f9c58a,Imperialism,"To better illustrate this idea, Bassett focuses his analysis of the role of nineteenth-century maps during the ""scramble for Africa"". He states that maps ""contributed to empire by promoting, assisting, and legitimizing the extension of French and British power into West Africa"". During his analysis of nineteenth-century cartographic techniques, he highlights the use of blank space to denote unknown or unexplored territory. This provided incentives for imperial and colonial powers to obtain ""information to fill in blank spaces on contemporary maps"".",bassett focuses on what to illustrate his idea?,"List(List(76, 76, 64, 64, 64), List(nineteenth-century maps, nineteenth-century maps, the role of nineteenth-century maps, the role of nineteenth-century maps, the role of nineteenth-century maps during the ""scramble for Africa""))"
573092088ab72b1400f9c598,Imperialism,"Imperialism has played an important role in the histories of Japan, Korea, the Assyrian Empire, the Chinese Empire, the Roman Empire, Greece, the Byzantine Empire, the Persian Empire, the Ottoman Empire, Ancient Egypt, the British Empire, India, and many other empires. Imperialism was a basic component to the conquests of Genghis Khan during the Mongol Empire, and of other war-lords. Historically recognized Muslim empires number in the dozens. Sub-Saharan Africa has also featured dozens of empires that predate the European colonial era, for example the Ethiopian Empire, Oyo Empire, Asante Union, Luba Empire, Lunda Empire, and Mutapa Empire. The Americas during the pre-Columbian era also had large em


## Task 1: Develop a LLM Pipeline

Create a language model pipeline that efficiently answers questions by leveraging pre-trained model.

###1.1: Create a Hugging Face Q&A Pipeline
Initialize a QA pipeline using a specified model tailored for question answering. This step involves selecting a model that has been optimized for the "`question-answering`" task.

**distilbert-base-cased-distilled-squad modl: https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad**

For this notebook we'll use the <a href="https://huggingface.co/docs/transformers/v4.36.1/main_classes/pipelines" target="_blank">Hugging Face Pipeline</a> to use models for inference. 

In [0]:
##
## Import the pipeline function from the transformers library
from transformers import pipeline  
## Define variables for the model name, device mapping, and cache directory
hf_model_name = "distilbert-base-cased-distilled-squad"  
device_map = "auto"  ## Automatically use the best available device (CPU or GPU)
cache_dir = "/hf_cache" ## Path for caching data

## Initialize a question-answering pipeline with the specified model
qa_pipeline = pipeline(
    task="question-answering",  ## Specify the task type as 'question-answering'
    model=hf_model_name,  ## Model to be loaded
    model_kwargs={"cache_dir": cache_dir},  
)

2025-11-15 23:22:06.677956: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-15 23:22:06.681717: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-15 23:22:06.734544: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-15 23:22:07.529524: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]


###1.2: Test Question-Answering Pipeline
Validate the pipeline's functionality by running a predefined question and context to observe how the model interprets and responds.


**Refer to this link to get all the neccessary Parameters**
https://huggingface.co/docs/transformers/v4.36.1/en/main_classes/pipelines#transformers.QuestionAnsweringPipeline

In [0]:
##
## Define the context string where the model will search for answers
context = """Marie Curie was a Polish and naturalized-French physicist and chemist who conducted pioneering research on radioactivity. She was the first woman to win a Nobel Prize and the first person and only woman to win the Nobel prize twice in different scientific fields."""

## Define the question to be answered based on the given context
question = "Why is Marie Curie famous?"

## Use the question-answering pipeline to find an answer to the question from the context
answer = qa_pipeline(context=context, question=question, token_type_ids=None) # question (str or List[str]) , context (str or List[str]) 
## Print the question and answer
print(f"Question: {question}")

print(f"Answer: {answer}")
print("===============================================")

## Print the context to show the content the model used to find the answer
print(f"Context: {context}")

Question: Why is Marie Curie famous?
Answer: {'score': 0.17066407203674316, 'start': 74, 'end': 120, 'answer': 'conducted pioneering research on radioactivity'}
Context: Marie Curie was a Polish and naturalized-French physicist and chemist who conducted pioneering research on radioactivity. She was the first woman to win a Nobel Prize and the first person and only woman to win the Nobel prize twice in different scientific fields.


In [0]:
# Another test
## Define the context string where the model will search for answers
context2 = """On March 17, 1752, the Governor-General of New France, Marquis de la Jonquière, died and was temporarily replaced by Charles le Moyne de Longueuil. His permanent replacement, the Marquis Duquesne, did not arrive in New France until 1752 to take over the post. The continuing British activity in the Ohio territories prompted Longueuil to dispatch another expedition to the area under the command of Charles Michel de Langlade, an officer in the Troupes de la Marine. Langlade was given 300 men, including French-Canadians and warriors of the Ottawa. His objective was to punish the Miami people of Pickawillany for not following Céloron's orders to cease trading with the British. On June 21, the French war party attacked the trading centre at Pickawillany, capturing three traders and killing 14 people of the Miami nation, including Old Briton. He was reportedly ritually cannibalized by some aboriginal members of the expedition.	"""

## Define the question to be answered based on the given context
question2 = "What Governor in charge of New France died in 1752?"

## Use the question-answering pipeline to find an answer to the question from the context
answer2 = qa_pipeline(context=context2, question=question2, token_type_ids=None) # question (str or List[str]) , context (str or List[str]) 
## Print the question and answer
print(f"Question: {question2}")

print(f"Answer: {answer2}")
print("===============================================")

## Print the context to show the content the model used to find the answer
print(f"Context: {context2}")

Question: What Governor in charge of New France died in 1752?
Answer: {'score': 0.9952384829521179, 'start': 55, 'end': 78, 'answer': 'Marquis de la Jonquière'}
Context: On March 17, 1752, the Governor-General of New France, Marquis de la Jonquière, died and was temporarily replaced by Charles le Moyne de Longueuil. His permanent replacement, the Marquis Duquesne, did not arrive in New France until 1752 to take over the post. The continuing British activity in the Ohio territories prompted Longueuil to dispatch another expedition to the area under the command of Charles Michel de Langlade, an officer in the Troupes de la Marine. Langlade was given 300 men, including French-Canadians and warriors of the Ottawa. His objective was to punish the Miami people of Pickawillany for not following Céloron's orders to cease trading with the British. On June 21, the French war party attacked the trading centre at Pickawillany, capturing three traders and killing 14 people of the Miami nation, incl

In [0]:
#  Another test
## Define the context string where the model will search for answers
context2 = """During the First Sino-Japanese War in 1894, Japan absorbed Taiwan. As a result of the Russo-Japanese War in 1905, Japan took part of Sakhalin Island from Russia. Korea was annexed in 1910. During World War I, Japan took German-leased territories in China’s Shandong Province, as well as the Mariana, Caroline, and Marshall Islands. In 1918, Japan occupied parts of far eastern Russia and parts of eastern Siberia as a participant in the Siberian Intervention. In 1931 Japan conquered Manchuria from China. During the Second Sino-Japanese War in 1937, Japan's military invaded central China and by the end of the Pacific War, Japan had conquered much of the Far East, including Hong Kong, Vietnam, Cambodia, Myanmar, the Philippines, Indonesia, part of New Guinea and some islands of the Pacific Ocean. Japan also invaded Thailand, pressuring the country into a Thai/Japanese alliance. Its colonial ambitions were ended by the victory of the United States in the Second World War and the following treaties which remanded those territories to American administration or their original owners.	"""

## Define the question to be answered based on the given context
question2 = "Which country did Japan force into an alliance?"

## Use the question-answering pipeline to find an answer to the question from the context
answer2 = qa_pipeline(context=context2, question=question2, token_type_ids=None) # question (str or List[str]) , context (str or List[str]) 
## Print the question and answer
print(f"Question: {question2}")

print(f"Answer: {answer2}")
print("===============================================")

## Print the context to show the content the model used to find the answer
print(f"Context: {context2}")

Question: Which country did Japan force into an alliance?
Answer: {'score': 0.2483377903699875, 'start': 861, 'end': 874, 'answer': 'Thai/Japanese'}
Context: During the First Sino-Japanese War in 1894, Japan absorbed Taiwan. As a result of the Russo-Japanese War in 1905, Japan took part of Sakhalin Island from Russia. Korea was annexed in 1910. During World War I, Japan took German-leased territories in China’s Shandong Province, as well as the Mariana, Caroline, and Marshall Islands. In 1918, Japan occupied parts of far eastern Russia and parts of eastern Siberia as a participant in the Siberian Intervention. In 1931 Japan conquered Manchuria from China. During the Second Sino-Japanese War in 1937, Japan's military invaded central China and by the end of the Pacific War, Japan had conquered much of the Far East, including Hong Kong, Vietnam, Cambodia, Myanmar, the Philippines, Indonesia, part of New Guinea and some islands of the Pacific Ocean. Japan also invaded Thailand, pressuring 

## Task 2: Model Development and Registering
Track the developed model using MLflow and register it in the Unity Catalog for lifecycle management.

### 2.1: Track LLM Development with MLflow

Log the model's parameters, configuration, and outputs to MLflow for tracking experiments, versioning, and reproducibility.

In [0]:
# ##
# ## Import necessary MLflow and related library modules for model tracking
# import mlflow
# from mlflow.models import infer_signature
# from mlflow.transformers import generate_signature_output

# ## Generate a model output using the QA pipeline for a given input to use in the model signature
# output = generate_signature_output(qa_pipeline, {"context" : context, "question" : question})

# ## Infer a model signature that defines the input and output schema of the model
# signature = infer_signature({"context" : context, "question" : question}, output)

# ## Set the name of the experiment in MLflow
# experiment_name = f"/Users/{DA.username}/GenAI-As-04-Batch-Demo"
# mlflow.set_experiment(experiment_name)

# ## Define a path within the MLflow Artifacts repository to store the model
# model_artifact_path = "qa_pipeline"

# ## Start an MLflow run to log parameters, artifacts, and models
# with mlflow.start_run():
#     ## Log parameters used in the model; here, the model name
#     mlflow.log_params({ "hf_model_name":hf_model_name,})

#     ## Define inference configuration for logging purposes, could include other configurations
#     inference_configuration={ "hf_model_name":hf_model_name,}

#     ## Log the model along with its configuration, signature, and an example for use
#     model_info = mlflow.transformers.log_model(
#         transformers_model=hf_model_name,
#         artifact_path=model_artifact_path,
#         task="question-answering",  # Type of task for the model
#         inference_config=inference_configuration,  ## Configuration used for inference
#         signature=signature,  ## Signature that defines model input and output
#         input_example={"question": "Why is Marie Curie famous?", "context": context},  ## Example of input
#     )

2025/11/15 23:45:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-adde1550-8a14.cloud.databricks.com/ml/experiments/4117165248792101/models/m-ff4d2f7e55354718a4c8aeab2a071122?o=253704115258622


---------------------------------------------------------------------------
MlflowException                           Traceback (most recent call last)
File <command-5151829411170670>, line 29
     26 inference_configuration={ "hf_model_name":hf_model_name,}
     28 ## Log the model along with its configuration, signature, and an example for use
---> 29 model_info = mlflow.transformers.log_model(
     30     transformers_model=hf_model_name,
     31     artifact_path=model_artifact_path,
     32     task="question-answering",  # Type of task for the model
     33     inference_config=inference_configuration,  ## Configuration used for inference
     34     signature=signature,  ## Signature that defines model input and output
     35     input_example={"question": "Why is Marie Curie famous?", "context": context},  ## Example of input
     36 )

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-a236b83d-2163-47f2-8358-0e635cdd52f5/lib/python3.11/site-packages/mlflow/transformers/__init__

In [0]:
##

## Import necessary MLflow and related library modules for model tracking
import mlflow
from mlflow.models import infer_signature
from mlflow.transformers import generate_signature_output

## Generate a model output using the QA pipeline for a given input to use in the model signature
output = generate_signature_output(qa_pipeline, {"question": question, "context": context})

## Infer a model signature that defines the input and output schema of the model
signature = infer_signature({"question": question, "context": context}, output)

## Set the name of the experiment in MLflow
experiment_name = f"/Users/{DA.username}/GenAI-As-04-Batch-Demo"
mlflow.set_experiment(experiment_name)

## Define a path within the MLflow Artifacts repository to store the model
model_artifact_path = "qa_pipeline"

## Start an MLflow run to log parameters, artifacts, and models
with mlflow.start_run():
    ## Log parameters used in the model; here, the model name
    mlflow.log_params({
        "hf_model_name": hf_model_name,
    })

    ## Define inference configuration for logging purposes, could include other configurations
    inference_config = {
        "hf_model_name": hf_model_name,
    }

    ## Log the model along with its configuration, signature, and an example for use
    model_info = mlflow.transformers.log_model(
        transformers_model=qa_pipeline,
        artifact_path=model_artifact_path,
        task="question-answering",  ## Type of task for the model
        inference_config=inference_config,  ## Configuration used for inference
        signature=signature,  ## Signature that defines model input and output
        input_example={"question": "Why is Marie Curie famous?", "context": context},  ## Example of input
    )
    

2025/11/15 23:46:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[2025-11-15 23:46:38,241] [WARNING] [real_accelerator.py:162:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.
[2025-11-15 23:46:38,243] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cpu (auto detect)


🔗 View Logged Model at: https://dbc-adde1550-8a14.cloud.databricks.com/ml/experiments/4117165248792101/models/m-f0ab5a45ff87443da23b0ef95a8ea887?o=253704115258622


README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

### 2.2: Query the MLflow Tracking Server
Retrieve information about the model's performance and other metrics from the MLflow tracking server.

In [0]:
##
## Retrieve the experiment ID using the experiment name
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
## Search for all runs in the experiment using the experiment ID
runs = mlflow.search_runs([experiment_id])
## Sort the runs by their start time in descending order and get the run ID of the latest run
last_run_id = runs.sort_values('start_time', ascending=False).iloc[0].run_id
## Construct the model URI using the last run ID and the specified artifact path
model_uri = f"runs:/{last_run_id}/{model_artifact_path}"
model_uri

'runs:/37fd6a1ebdfe427495753ac10ba0f996/qa_pipeline'

###2.3: Load Model Back as a Pipeline
Load the registered model from MLflow to verify its performance and integration capabilities post-registration.


In [0]:
##
loaded_qa_pipeline = mlflow.pyfunc.load_model(model_uri=model_uri)
loaded_qa_pipeline.predict({"question": question, "context": context})

['conducted pioneering research on radioactivity']

In [0]:
loaded_qa_pipeline = mlflow.pyfunc.load_model(model_uri=model_uri)
loaded_qa_pipeline.predict({"question": question2, "context": context2})

['Thai/Japanese']

### 2.4: Register the Model to Unity Catalog
Register the model in the Unity Catalog for better version control and to facilitate the deployment process.

In [0]:
##
from mlflow import MlflowClient
## Define the model name
model_name = f"{DA.catalog_name}.{DA.schema_name}.qa_pipeline"
## Set the MLflow registry URI
mlflow.set_registry_uri("databricks-uc")
## Register the model in the MLflow model registry under the specified name and model URI
mlflow.register_model(model_uri=model_uri, name=model_name)

Successfully registered model 'dbacademy.labuser12678309_1763243239.qa_pipeline'.
2025/11/15 23:50:08 WARNING mlflow.tracking._model_registry.fluent: Run with id 37fd6a1ebdfe427495753ac10ba0f996 has no artifacts at artifact path 'qa_pipeline', registering model based on models:/m-f0ab5a45ff87443da23b0ef95a8ea887 instead


Uploading artifacts:   0%|          | 0/20 [00:00<?, ?it/s]

🔗 Created version '1' of model 'dbacademy.labuser12678309_1763243239.qa_pipeline': https://dbc-adde1550-8a14.cloud.databricks.com/explore/data/models/dbacademy/labuser12678309_1763243239/qa_pipeline/version/1?o=253704115258622


<ModelVersion: aliases=[], creation_timestamp=1763250615633, current_stage=None, deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1763250617865, metrics=[], model_id='m-f0ab5a45ff87443da23b0ef95a8ea887', name='dbacademy.labuser12678309_1763243239.qa_pipeline', params=[<LoggedModelParameter: key='hf_model_name', value='distilbert-base-cased-distilled-squad'>], run_id='37fd6a1ebdfe427495753ac10ba0f996', run_link=None, source='models:/m-f0ab5a45ff87443da23b0ef95a8ea887', status='READY', status_message='', tags={}, user_id='labuser12678309_1763243239@vocareum.com', version='1'>

## Task 3: LLM Model State Management
In this task, you'll manage your model's lifecycle across different stages using MLflow and Unity Catalog. By leveraging MLflow's Model Registry, you will update and maintain the model's state to enhance tracking, version control, and deployment efficiency.

###3.1: Search and Inspect Registered Model
Identify and inspect the latest version of your registered model to ensure you are managing the most current and relevant iteration. This step is crucial as it determines the baseline for setting model stages or aliases.

- Retrieve the Latest Model Version
- Set Model Alias

In [0]:
##
def get_latest_model_version(model_name_in):
    ## Initialize the MLflow Client to interact with the MLflow server
    client = MlflowClient()
    
    ## Search for all versions of the specified model in the Model Registry
    model_version_infos = client.search_model_versions("name = '%s'" % model_name_in)
    
    ## Extract the version numbers and return the highest (latest) version
    return max([model_version_info.version for model_version_info in model_version_infos])

## Initialize the MLflow Client for further operations
client = mlflow.tracking.MlflowClient()

## Get the latest version number of the specified model
current_model_version = get_latest_model_version(model_name)

## Set an alias 'champion' for the latest version of the model
client.set_registered_model_alias(
  name=model_name, alias="champion",
  version=current_model_version
  )

## Task 4: Batch Inference
Perform inference using the registered model on new data, both in single-node and multi-node environments.

###4.1: Load the Model for Batch Inference
Prepare the environment and load the model from Unity Catalog for batch processing.

In [0]:
prod_data_table = f"{DA.catalog_name}.{DA.schema_name}.m4_1_lab_prod_data"
## Read data from the specified Spark table and limit the results to the first 100 rows
prod_data_df = spark.read.table(prod_data_table).limit(100)
## Display the DataFrame to visualize the top 100 rows of the dataset
display(prod_data_df.limit(10))

id,title,context,question,answers
56be4db0acb8001400a502ec,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the ""golden anniversary"" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as ""Super Bowl L""), so that the logo could prominently feature the Arabic numerals 50.",Which NFL team represented the AFC at Super Bowl 50?,"List(List(177, 177, 177), List(Denver Broncos, Denver Broncos, Denver Broncos))"
56be4db0acb8001400a502ed,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the ""golden anniversary"" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as ""Super Bowl L""), so that the logo could prominently feature the Arabic numerals 50.",Which NFL team represented the NFC at Super Bowl 50?,"List(List(249, 249, 249), List(Carolina Panthers, Carolina Panthers, Carolina Panthers))"
56be4db0acb8001400a502ee,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the ""golden anniversary"" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as ""Super Bowl L""), so that the logo could prominently feature the Arabic numerals 50.",Where did Super Bowl 50 take place?,"List(List(403, 355, 355), List(Santa Clara, California, Levi's Stadium, Levi's Stadium in the San Francisco Bay Area at Santa Clara, California.))"
56be4db0acb8001400a502ef,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the ""golden anniversary"" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as ""Super Bowl L""), so that the logo could prominently feature the Arabic numerals 50.",Which NFL team won Super Bowl 50?,"List(List(177, 177, 177), List(Denver Broncos, Denver Broncos, Denver Broncos))"
56be4db0acb8001400a502f0,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the c

###4.2: Single-node Batch Inference
Conduct inference tests on a limited dataset to validate the model's response accuracy and speed in a single-node setup.


In [0]:
display(prod_data_df.limit(10))

id,title,context,question,answers
56be4db0acb8001400a502ec,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the ""golden anniversary"" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as ""Super Bowl L""), so that the logo could prominently feature the Arabic numerals 50.",Which NFL team represented the AFC at Super Bowl 50?,"List(List(177, 177, 177), List(Denver Broncos, Denver Broncos, Denver Broncos))"
56be4db0acb8001400a502ed,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the ""golden anniversary"" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as ""Super Bowl L""), so that the logo could prominently feature the Arabic numerals 50.",Which NFL team represented the NFC at Super Bowl 50?,"List(List(249, 249, 249), List(Carolina Panthers, Carolina Panthers, Carolina Panthers))"
56be4db0acb8001400a502ee,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the ""golden anniversary"" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as ""Super Bowl L""), so that the logo could prominently feature the Arabic numerals 50.",Where did Super Bowl 50 take place?,"List(List(403, 355, 355), List(Santa Clara, California, Levi's Stadium, Levi's Stadium in the San Francisco Bay Area at Santa Clara, California.))"
56be4db0acb8001400a502ef,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the ""golden anniversary"" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as ""Super Bowl L""), so that the logo could prominently feature the Arabic numerals 50.",Which NFL team won Super Bowl 50?,"List(List(177, 177, 177), List(Denver Broncos, Denver Broncos, Denver Broncos))"
56be4db0acb8001400a502f0,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the c

In [0]:
##
## Load the latest version of the model from MLflow using the provided model URI
latest_model = mlflow.pyfunc.load_model(model_uri=f"models:/{model_name}/{current_model_version}")
print(f"Latest model : {latest_model}")

## Convert the first two rows of the DataFrame to a Pandas DataFrame for easier manipulation
prod_data_sample_pdf = prod_data_df.limit(2).toPandas()

## Define a list of questions to be answered by the model
questions = ["What color was used to emphasize the 50th anniversary of the Super Bowl?"]

## Generate answers for each question by applying the loaded model on the context provided in the DataFrame
qa_results = [latest_model.predict({"question": q, "context": doc}) for q, doc in zip(questions, prod_data_sample_pdf["context"])]

## Import the pprint function for formatted display of objects
from pprint import pprint

## Print each result in a formatted manner using pprint for better readability
pprint(qa_results)

Latest model : mlflow.pyfunc.loaded_model:
  artifact_path: dbfs:/databricks/mlflow-tracking/4117165248792101/logged_models/m-f0ab5a45ff87443da23b0ef95a8ea887/artifacts
  flavor: mlflow.transformers
  run_id: 37fd6a1ebdfe427495753ac10ba0f996

[['gold']]


###4.3: Multi-node Batch Inference
Scale the inference process using Spark to simulate real-world, large-scale data handling scenarios.


In [0]:
##
from pyspark.sql.functions import col

## Ensure that the input DataFrame contains 'question' and 'context' columns
prod_data_df = prod_data_df.withColumn("question", col("question"))
prod_data_df = prod_data_df.withColumn("context", col("context"))

prod_model_udf = mlflow.pyfunc.spark_udf(
    spark,
    model_uri=f"models:/{model_name}@champion",
    env_manager="local",
    result_type="string",
)

batch_inference_results_df = prod_data_df.withColumn("generated_answer", prod_model_udf("question", "context"))

## Display the DataFrame containing the results of the batch inference with generated answers
batch_inference_results_df.display()

2025/11/16 00:01:08 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2025/11/16 00:01:09 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


id,title,context,question,answers,generated_answer
56be4db0acb8001400a502ec,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the ""golden anniversary"" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as ""Super Bowl L""), so that the logo could prominently feature the Arabic numerals 50.",Which NFL team represented the AFC at Super Bowl 50?,"List(List(177, 177, 177), List(Denver Broncos, Denver Broncos, Denver Broncos))",Denver Broncos
56be4db0acb8001400a502ed,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the ""golden anniversary"" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as ""Super Bowl L""), so that the logo could prominently feature the Arabic numerals 50.",Which NFL team represented the NFC at Super Bowl 50?,"List(List(249, 249, 249), List(Carolina Panthers, Carolina Panthers, Carolina Panthers))",Carolina Panthers
56be4db0acb8001400a502ee,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the ""golden anniversary"" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as ""Super Bowl L""), so that the logo could prominently feature the Arabic numerals 50.",Where did Super Bowl 50 take place?,"List(List(403, 355, 355), List(Santa Clara, California, Levi's Stadium, Levi's Stadium in the San Francisco Bay Area at Santa Clara, California.))","Levi's Stadium in the San Francisco Bay Area at Santa Clara, California"
56be4db0acb8001400a502ef,Super_Bowl_50,"Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the ""golden anniversary"" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as ""Super Bowl L""), so that the logo could prominently feature the Arabic numerals 50.",Which NFL team won Super Bowl 50?,"List(List(177, 177, 177), List(Denver Broncos, Denver Bronc

###4.4: Write Inference Results to Delta Table
Store the inference results in a Delta table to ensure data integrity and enable further analysis.



In [0]:
prod_data_summaries_table_name = f"{DA.catalog_name}.{DA.schema_name}.m4_1_lab_batch_inference"
batch_inference_results_df.write.mode("append").saveAsTable(prod_data_summaries_table_name)

##Task 5: Batch Inference Using `ai_query()`

Utilize SQL capabilities to perform batch inference directly using SQL queries, integrating AI functions for broader accessibility and efficiency.

### 5.1: Run SQL Batch Inference

Create a SQL query that executes an AI model inference directly within the SQL. This approach utilizes the `ai_query()` function in SQL to process batch queries against the dataset.


In [0]:
# This code gets all serving models using the Databricks SDK 
from databricks.sdk import WorkspaceClient

# Create a workspace client
w = WorkspaceClient()

# List all model serving endpoints
endpoints = w.serving_endpoints.list()

# Print the names of all serving models
for endpoint in endpoints:
    print(endpoint.name)

my_qa
databricks-gpt-5
databricks-gemini-2-5-flash
databricks-claude-sonnet-4-5
databricks-gpt-oss-120b
databricks-gpt-5-mini
databricks-gpt-5-nano
databricks-gemini-2-5-pro
databricks-gpt-oss-20b
databricks-qwen3-next-80b-a3b-instruct
databricks-llama-4-maverick
databricks-gemma-3-12b
databricks-meta-llama-3-1-8b-instruct
databricks-meta-llama-3-3-70b-instruct
databricks-claude-opus-4-1
databricks-claude-sonnet-4
databricks-claude-3-7-sonnet
databricks-gte-large-en
databricks-bge-large-en
databricks-meta-llama-3-1-405b-instruct
databricks-claude-opus-4


###Step 1: Run SQL Batch Inference

In [0]:
%sql
CREATE OR REPLACE TABLE ai_query_inference AS (
  SELECT
    id,
    question,
    answers,
    ai_query(
      "databricks-claude-opus-4-1", 
      CONCAT("Asking question: ", question, " Answer: ", CAST(answers AS STRING))
    ) as generated_answer
  FROM m4_1_lab_prod_data LIMIT 100
);

num_affected_rows,num_inserted_rows


###5.2: Query Inference Results
Query the generated table to view the inference results.

In [0]:
%sql
---- Retrieve all records from the 'ai_query_inference' table to view the results
SELECT * FROM ai_query_inference

id,question,answers,generated_answer
56be4db0acb8001400a502ec,Which NFL team represented the AFC at Super Bowl 50?,"List(List(177, 177, 177), List(Denver Broncos, Denver Broncos, Denver Broncos))","Based on the information provided, the **Denver Broncos** represented the AFC at Super Bowl 50. The Broncos defeated the Carolina Panthers 24-10 in Super Bowl 50, which was played on February 7, 2016, at Levi's Stadium in Santa Clara, California. This was the Broncos' eighth Super Bowl appearance and their third championship victory."
56be4db0acb8001400a502ed,Which NFL team represented the NFC at Super Bowl 50?,"List(List(249, 249, 249), List(Carolina Panthers, Carolina Panthers, Carolina Panthers))","Based on the information provided, the **Carolina Panthers** represented the NFC at Super Bowl 50. Super Bowl 50 was played on February 7, 2016, between the Carolina Panthers (NFC) and the Denver Broncos (AFC). The Panthers had an excellent season, finishing 15-1 in the regular season and defeating the Arizona Cardinals in the NFC Championship Game to earn their spot in Super Bowl 50."
56be4db0acb8001400a502ee,Where did Super Bowl 50 take place?,"List(List(403, 355, 355), List(Santa Clara, California, Levi's Stadium, Levi's Stadium in the San Francisco Bay Area at Santa Clara, California.))","Based on the information provided, Super Bowl 50 took place at **Levi's Stadium in Santa Clara, California**, which is located in the San Francisco Bay Area."
56be4db0acb8001400a502ef,Which NFL team won Super Bowl 50?,"List(List(177, 177, 177), List(Denver Broncos, Denver Broncos, Denver Broncos))","Looking at the provided answer format, I can see that Super Bowl 50 was won by the **Denver Broncos**. The Broncos defeated the Carolina Panthers 24-10 on February 7, 2016, at Levi's Stadium in Santa Clara, California. This was the Broncos' third Super Bowl championship."
56be4db0acb8001400a502f0,What color was used to emphasize the 50th anniversary of the Super Bowl?,"List(List(488, 488, 521), List(gold, gold, gold))","Based on the answer provided, **gold** was the color used to emphasize the 50th anniversary of the Super Bowl. This was for Super Bowl 50, which took place in 2016. Instead of using the traditional Roman numerals (which would have been ""L""), the NFL chose to use the number ""50"" and emphasized it with gold coloring throughout their branding and marketing materials for this milestone anniversary game."
56be8e613aeaaa14008c90d1,What was the theme of Super Bowl 50?,"List(List(487, 521, 487), List(""golden anniversary"", gold-themed, ""golden anniversary))","Based on the provided answer format, the theme of Super Bowl 50 was the **""golden anniversary""** or **gold-themed** celebration. Super Bowl 50 took place in 2016, marking the 50th championship game in NFL history. The NFL chose to commemorate this milestone with a gold theme throughout the season and especially during the championship game. This included gold-accented logos, gold numbering (using ""50"" instead of the traditional Roman numeral ""L""), and various golden anniversary celebrations and ceremonies throughout the event."
56be8e613aeaaa14008c90d2,What day was the game played on?,"List(List(334, 334, 334), List(February 7, 2016, February 7, February 7, 2016))","Based on the answer provided, the game was played on **February 7, 2016**."
56be8e613aeaaa14008c90d3,What is the AFC short for?,"List(List(133, 133, 133), List(American Football Conference, American Football Conference, American Football Conference))","Based on the answer provided, AFC stands for **American Football Conference**. This is one of the two conferences in the National Football League (NFL), along with the NFC (National Football Conference). The AFC was formed in 1970 when the NFL merged with the American Football League (AFL)."
56bea9923aeaaa14008c91b9,What was the theme of Super Bowl 50?,"List(List(487, 521, 521), List(""golden anniversary"", gold-themed, gold))","Based on the answer provided, the theme

## Conclusion

In this lab, you successfully implemented a batch inference workflow using a small language model. You created a question-answering pipeline, tracked and registered the model using MLflow, managed model versions and stages with Unity Catalog, and performed both single-node and multinode batch inference. Finally, you explored an alternative method for batch inference using the `ai_query` SQL function.

&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>